# MapBiomas Fire Monitor — Pipeline (M0-M6)

## Documentation and User Guide
Expand this section to understand the data architecture, environment requirements, and workflow.

#### Introduction and Context

This pipeline is the automated processing core for the **MapBiomas Fire Scar Monitoring**. It is organized into sequential stages (M0-M8), each with a specific responsibility:

* **M0 -- Setup & Configuration**: Colab-native setup. Clones the repository, installs dependencies, configures GCS and GEE project paths via `set_global_opts()`, authenticates with Earth Engine.

* **M1 & M2 -- Export + Mosaic**: M1 exports quality-mosaicked satellite images from GEE to GCS as GeoTIFF chunks. M2 assembles chunks into Cloud-Optimized GeoTIFFs using GDAL. Handles Sentinel-2 (+ Landsat, MODIS, HLS in future). Supports monthly/annual periods with configurable mosaics (minnbr, minnbr_buffer).

* **M3 -- Sample Collection**: GEE JavaScript Toolkit. Users draw fire (burned) and notFire (unburned) polygons on satellite imagery. Exports FeatureCollections (GEE) and CSV samples (GCS) with full metadata. Multi-country, multi-language (EN/ES/PT).

* **M4, M5 & M6 -- Train, Classify & Publish**: M4 trains a DNN classifier from M3 samples + M2 mosaics. M5 runs tile-by-tile inference (probability + dayOfYear per grid cell). M6 assembles tiles into regional mosaics, computes burned area statistics (CSV), and uploads results to GEE as ImageCollections.

* **M7 & M8 -- Under Development**: M7 will apply spatial/temporal masks. M8 will evaluate candidate versions and select the official release.

**Global Inputs:** Raw satellite image collections (GEE), training polygons (M3), auxiliary land cover maps.

**Global Outputs:** GeoTIFF chunks and COG mosaics (GCS), DNN model weights (GCS), int16 2-band classification rasters (GCS), consolidated statistics (CSV), versioned ImageCollections (GEE).

> Expand the sections below for detailed data architecture, naming rules, and path examples.


---

### M0 -- Project Configuration

Centralizes all project-level configuration via `set_global_opts()`. Defines where data is stored (GCS bucket + GEE project), which campaign is active, and pipeline parameters (sensor, periodicity, mosaic method). The region FeatureCollection is copied into the campaign at setup time.

**Key parameters:**
```
country                  country/region code
campaign                 active campaign (MONITOR_01, COLLECTION_01, ...)

gcs_bucket               GCS bucket name
gcs_library_images_prefix  where base images live in GCS
gcs_campaigns_prefix       where campaign outputs live in GCS

gee_project              GEE Cloud Project ID
gee_library_images_prefix  where base images live in GEE
gee_campaigns_prefix       where campaign outputs live in GEE
asset_regions            FeatureCollection with region geometries

sensor, periodicity      e.g. [sentinel2], [monthly]
mosaic_methods           e.g. [minnbr, minnbr_buffer]
```

> Both Colab temporary storage and local disk act as **ephemeral space**. Persistence always occurs in **Google Cloud Storage (GCS)** and **Google Earth Engine (GEE)**.


---

### M1 & M2 -- Export + Mosaic Assembly

M1 exports quality-mosaicked satellite images from GEE to GCS as GeoTIFF chunks. M2 assembles those chunks into full-region Cloud-Optimized GeoTIFFs (COGs) using GDAL. Both modules read and write under `LIBRARY_IMAGES`.

**GEE Assets** -- ImageCollections organised per band:
```
{gee_images_prefix}/LIBRARY_IMAGES/
  SENTINEL2/MONTHLY/MINNBR/
    blue/                    <-- ImageCollection (M1 exports)
      image_peru_fire_sentinel2_minnbr_blue_2025_08
    green/
      image_peru_fire_sentinel2_minnbr_green_2025_08
    red/
    nir/
    swir1/
    swir2/
    dayOfYear/
  ... also MINNBR_BUFFER/ for buffered mosaics
```

**GCS Paths** -- Chunks written by M1, COGs written by M2:
```
{gcs_images_prefix}/LIBRARY_IMAGES/
  SENTINEL2/MONTHLY/MINNBR/2025_08/
    CHUNKS/                                          <-- M1 writes
      image_peru_fire_sentinel2_minnbr_blue_2025_08_0000196608-0000131072.tif
      image_peru_fire_sentinel2_minnbr_nir_2025_08_0000196608-0000131072.tif
      ...
    COG/                                             <-- M2 writes
      image_peru_fire_sentinel2_minnbr_blue_2025_08_cog.tif
      image_peru_fire_sentinel2_minnbr_nir_2025_08_cog.tif
      ...
  ... also MINNBR_BUFFER/ for buffered mosaics
```

**Naming rules:**
- Chunk: `image_{country}_fire_{sensor}_{mosaic}_{band}_{date}_{tile_id}.tif`
- COG: `image_{country}_fire_{sensor}_{mosaic}_{band}_{date}_cog.tif`


---

### M3 -- Sample Collection

Training data collection via a custom GEE JavaScript Toolkit. Users draw fire (burned) and notFire (unburned) polygons on satellite imagery. Samples are exported as FeatureCollections to GEE and as CSV to GCS, with full metadata (satellite, date, region, campaign).

**GEE Assets** -- FeatureCollections with polygon geometries:
```
{gee_campaigns_prefix}/{campaign}/
  AUXILIARY/                                        <-- M0 setup
    regiones_fuego_peru_v1                           (FeatureCollection)

  LIBRARY_SAMPLES/                                   <-- M3 writes
    sample_0001_peru_r1_costa_norte_2025_05           (FeatureCollection)
    sample_0001_peru_r5_andes_sur_2025_12
    sample_0001_peru_r9_amazonia_baja_centro_2025_08
```

**GCS Paths** -- CSV copies for model training:
```
{gcs_campaigns_prefix}/{campaign}/
  LIBRARY_SAMPLES/                                   <-- M3 writes
    sample_0001_peru_r1_costa_norte_2025_05.csv
    sample_0001_peru_r5_andes_sur_2025_12.csv
    sample_0001_peru_r9_amazonia_baja_centro_2025_08.csv
```

**Naming rule:** `sample_{id}_{region}_{comment?}_{period}.csv`


---

### M4, M5 & M6 -- Train, Classify & Publish

M4 trains a Deep Neural Network classifier using pixel samples extracted from M2 mosaics with M3 polygons. M5 runs tile-by-tile inference generating int16 2-band rasters (probability 0-1000 + dayOfYear). M6 assembles tiles into regional mosaics, computes statistics, and publishes to GEE.

**GEE Assets** -- Final classified images per region:
```
{gee_campaigns_prefix}/{campaign}/
  LIBRARY_CLASSIFICATIONS/REGIONAL/                  <-- M6 uploads
    training_001_peru_r1_v1_sentinel2/                (ImageCollection)
      peru_r1_costa_norte_2025_05                      (Image: probability + dayOfYear)
      peru_r1_costa_norte_2025_12
    training_003_peru_r9_sentinel2/                   (ImageCollection)
      peru_r9_amazonia_baja_centro_2025_08
      ...
```

**GCS Paths** -- Models (M4), tiles (M5), mosaics and stats (M6):
```
{gcs_campaigns_prefix}/{campaign}/
  LIBRARY_MODELS/                                    <-- M4 writes
    training_001_peru_r1_v1_sentinel2/
      weights.npz
      metadata.json
      metrics.json
      workplan/                                      <-- M5 workplan
        archived/
        pending/

  LIBRARY_CLASSIFICATIONS/                           <-- M5 + M6 write
    training_001_peru_r1_v1_sentinel2/
      CLASSIFIED_TILES/                              <-- M5 output
        tile_peru_r1_costa_norte_SA-00000_2025_05.tif
      CLASSIFIED_REGION/                             <-- M6 mosaics
        region_peru_r1_costa_norte_training_001_peru_r1_v1_sentinel2_2025_05.tif
      STATS/                                         <-- M6 statistics
        stats_tile.csv
        stats_region.csv
```

**Naming rules:**
- Model: `training_{id}_{shortname}_{sensor}/`
- Tile: `tile_{region}_{cell}_{period}.tif`
- Mosaic: `region_{region}_{model}_{period}.tif`


---

### M7 & M8 -- Post-Processing & Release

Under development. M7 will apply spatial and temporal masks. M8 will evaluate candidate versions and select the official release.


## [M0] — Environment Setup
Clones the repository (first run only), installs system and Python dependencies (GDAL, gcsfs, rasterio, earthengine-api), configures GCS bucket and GEE project via `set_global_opts()`, authenticates with Earth Engine, and registers the region FeatureCollection.

In [ ]:
# M0 -- Colab Setup
CONTRY_SELECTED = 'peru'
import sys, os, subprocess, shutil

# -- Clone repository (always fresh) --
if os.path.exists("/content/fire_monitor"):
    shutil.rmtree("/content/fire_monitor")
    print("Old clone removed.")

subprocess.run(["git", "clone", "https://github.com/mapbiomas/peru-fire.git", "fire_monitor"])

# -- Install system dependencies --
subprocess.run(["apt-get", "update", "-qq"])
subprocess.run(["apt-get", "install", "-y", "-qq", "gdal-bin", "python3-gdal"])
subprocess.run(["pip", "install", "-q", "earthengine-api", "gcsfs", "rasterio", "scipy", "tqdm"])

# -- Add to Python path --
repo_path = "/content/fire_monitor/mapbiomas_fire_monitor/version_01/src/core"
if repo_path not in sys.path: sys.path.insert(0, repo_path)
os.chdir(repo_path)

# -- Configure project --
from M0_auth_config import set_global_opts, authenticate, print_config

set_global_opts(
    # === Active Campaign ===
    country=F'{CONTRY_SELECTED}',                            # country/region code (e.g. 'peru', 'bolivia')
    campaign='MONITOR_01',                                  # campaign (e.g. 'MONITOR_01', 'COLLECTION_01', 'COLLECTION_02')

    # === Pipeline Settings ===
    sensor=['sentinel2'],                                    # list: ['sentinel2', 'landsat', 'hls', 'modis']
    periodicity=['monthly'],                                 # list: ['monthly', 'yearly']
    mosaic_methods=['minnbr', 'minnbr_buffer'],              # list: ['minnbr', 'minnbr_buffer', 'median', 'minndvi']
    personal_task_flag='MONITOR',                        # prefix for GEE task names
    language='en',                                           # 'en', 'es', 'pt', 'fr', 'in'

    # === Google Cloud Storage ===
    gcs_bucket='mapbiomas-fire',                                                    # bucket name
    gcs_library_images_prefix=F'sudamerica/{CONTRY_SELECTED}/CATALOG_01',           # prefix for LIBRARY_IMAGES
    gcs_campaigns_prefix=F'sudamerica/{CONTRY_SELECTED}/CATALOG_01',                # prefix for campaign outputs

    # === Google Earth Engine ===
    gee_project=F'mapbiomas-{CONTRY_SELECTED}',                                                                                             # GEE Cloud Project ID
    gee_library_images_prefix=F'projects/mapbiomas-{CONTRY_SELECTED}/assets/FIRE/CATALOG_01',                                               # asset path for images
    gee_campaigns_prefix=F'projects/mapbiomas-{CONTRY_SELECTED}/assets/FIRE/CATALOG_01',                                                    # asset path for campaigns
    asset_regions=F'projects/mapbiomas-{CONTRY_SELECTED}/assets/FIRE/CATALOG_01/MONITOR_DEV/AUXILIARY/regiones_fuego_{CONTRY_SELECTED}_v1', # asset path for region geometries
)

authenticate()
print_config()

## [M1] — Export (GEE → GCS)
Multi-sensor satellite image export from GEE ImageCollections to GCS. Handles Sentinel-2 (and in future: Landsat (5/7/8/9), MODIS, and HLS) with sensor-specific radiometric corrections, cloud masking, and configurable mosaic types (MINNBR, MINNBR_BUFFER, etc.).

In [ ]:
from M1_export_ui import run_ui, start_export

ui_exporter = run_ui(years=[2025,2026])

In [ ]:
# Export trigger (GEE -> GCS)

start_export(ui_exporter)

## [M2] — Mosaic Assembly (COG)

Assembles exported chunks into full-region Cloud-Optimized GeoTIFFs using GDAL VRT virtual stacking. Produces compressed, tiled COGs with DEFLATE compression.

In [ ]:
from M2_mosaic_ui import run_ui, start_mosaic_assembly
ui_assembler = run_ui(years=[2025, 2026])

In [ ]:
# Mosaic trigger (Local/GCS COG)
start_mosaic_assembly(ui_assembler)

## [M3] — Sample Collection (GEE Toolkit)
Training data collection via a custom GEE JavaScript Toolkit. Users draw fire/notFire polygons on satellite imagery. Samples are exported to GEE Assets and GCS with full metadata (satellite, date, region, campaign).

In [ ]:
from M3_sample_ui import show_toolkit_links
show_toolkit_links()

## [M4] — DNN Training
Trains a Deep Neural Network classifier using a flexible band extraction matrix. Samples pixels from M2 mosaics using M3 polygons, trains a configurable DNN, generates t-SNE audit plots, and saves model weights and metadata to GCS.

In [ ]:
from M4_model_trainer import run_ui, start_training
ui_trainer = run_ui()

In [ ]:
# Training trigger
start_training(ui_trainer)

## [M5] — Tile-Based Classification
Tile-based burned area classification using a trained DNN model. Loads the model from GCS, retrieves the cell grid for the target region, classifies each tile independently, and generates per-tile classified rasters with local statistics. Results are consumed by M6 for mosaicking and publication.

In [ ]:
# 1. M5 Scheduling Interface
from M5_classifier_ui import run_m5_ui
ui_m5 = run_m5_ui(years=[2025, 2026], periodicity_active=["monthly"])

In [ ]:
# 2. M5 Async Processing Engine
# n_workers: auto (cpu_count). Force com: run_m5_workplan(n_workers=4, progress_callback=ui_m5._on_tile_progress)
from M5_classifier import run_m5_workplan
run_m5_workplan(progress_callback=ui_m5._on_tile_progress)

## [M6] — Mosaic, Statistics & Publication
Assembles classified tiles into regional mosaics, computes burned area statistics, and publishes results to Google Earth Engine as ImageCollections. Provides an interactive dashboard for monitoring batch progress, analytics, and coverage.

In [ ]:
# 3. M6 Scheduling Interface
from M6_ui import run_m6_ui
ui_m6 = run_m6_ui()

In [ ]:
# 2. M6 Publish (mosaic + stats + GEE upload)
# Runs only when classified tiles (COMPLETED) are available in GCS.
# upload_gee=True sends regional mosaics to Google Earth Engine.
from M6_publisher import run_m6_publish
run_m6_publish(upload_gee=True)

## [M7] — Post-Classification Masks
🔨 **Under development.** Will apply spatial and temporal masks to classified outputs before final publication.

In [ ]:
# 🔨 M7 — Post-Classification Masks (Under development)
# Apply spatial and temporal masks to classified outputs.
# from M7_masks import run_m7_masks

## [M8] — Final Version Evaluation & Official Release
🔨 **Under development.** Will evaluate candidate versions and select the official release for publication.

In [ ]:
# 🔨 M8 — Final Version Evaluation & Official Release (Under development)
# Evaluate candidate versions and select the official release.
# from M8_release import run_m8_release